# Colab API

Runs Model 1, Model 2, or both from Google Colab.

In [ ]:
# 1. Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install packages

import os
import sys

PIP_CACHE_DIR = '/content/drive/MyDrive/Colab Notebooks/Customer_retention_analytics/pip_cache'
os.makedirs(PIP_CACHE_DIR, exist_ok=True)

print('Pip cache:', PIP_CACHE_DIR)

# Do not reinstall numpy/scipy/pandas in Colab. It can break compiled packages.
!"{sys.executable}" -m pip -q install --cache-dir "{PIP_CACHE_DIR}" fastapi uvicorn nest_asyncio joblib diskcache jinja2 typing-extensions
!"{sys.executable}" -m pip -q install --cache-dir "{PIP_CACHE_DIR}" --no-deps xgboost
!"{sys.executable}" -m pip -q install --cache-dir "{PIP_CACHE_DIR}" --only-binary=:all: --prefer-binary --no-deps llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [ ]:
# 3. Config

from pathlib import Path

USE_MODEL1 = True
USE_MODEL2 = True
UPLOAD_IF_MISSING = True

MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Customer_retention_analytics')

MODEL1_PATH = str(MODEL_DIR / 'xgboost_model.json')
CALIBRATOR_PATH = str(MODEL_DIR / 'isotonic_calibrator.joblib')
METADATA_PATH = str(MODEL_DIR / 'model_metadata.json')
MODEL2_PATH = str(MODEL_DIR / 'model2_retention_0.5b.gguf')

API_PORT = 8000

MODEL2_N_CTX = 1024
MODEL2_N_THREADS = 2
MODEL2_N_BATCH = 128
MODEL2_MAX_TOKENS = 256
MODEL2_TEMPERATURE = 0.3

print('Model folder:', MODEL_DIR)

In [ ]:
# 4. Imports

import json
import re
import stat
import subprocess
import threading
import time
import urllib.request
from typing import Any, Optional

import joblib
import nest_asyncio
import numpy as np
import pandas as pd
import uvicorn
import xgboost as xgb
from fastapi import FastAPI, HTTPException
from llama_cpp import Llama
from pydantic import BaseModel
from xgboost import XGBClassifier

MODEL1 = None
CALIBRATOR = None
METADATA = None
MODEL2 = None
MODEL2_LOCK = threading.Lock()

In [ ]:
# 5. File helper

def upload_file_if_missing(path):
    path = Path(path)
    if path.exists():
        print(f'Found file: {path.name}')
        return str(path)
    if not UPLOAD_IF_MISSING:
        raise FileNotFoundError(f'Missing file: {path}')

    from google.colab import files
    print(f'Upload missing file: {path.name}')
    uploaded = files.upload()
    if path.name not in uploaded:
        raise FileNotFoundError(f'Uploaded files did not include {path.name}')

    path.parent.mkdir(parents=True, exist_ok=True)
    os.replace(path.name, path)
    return str(path)

In [ ]:
# 6. Model 1 helpers

FORBIDDEN_INPUT_FIELDS = {
    'customer_id', 'customer_name', 'snapshot_date', 'loyalty',
    'customer_yearly_value', 'complaint_text', 'churn_flag'
}

def json_safe(value: Any) -> Any:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value

def prepare_feature_row(customer_data: dict, metadata: dict) -> pd.DataFrame:
    features = metadata['features']
    values = {feature: np.nan if customer_data[feature] is None else customer_data[feature] for feature in features}
    row = pd.DataFrame([values])
    for feature in metadata['numerical_features']:
        row[feature] = pd.to_numeric(row[feature], errors='raise')
    for feature in metadata['categorical_features']:
        row[feature] = row[feature].astype('category')
    return row

def validate_model1_input(customer_data: dict, features: list[str]):
    input_fields = set(customer_data)
    missing = sorted(set(features) - input_fields)
    unknown = sorted(input_fields - set(features))
    forbidden = sorted(input_fields & FORBIDDEN_INPUT_FIELDS)
    if missing:
        raise ValueError(f'Missing required fields: {missing}')
    if forbidden:
        raise ValueError(f'Forbidden fields: {forbidden}')
    if unknown:
        raise ValueError(f'Unknown fields: {unknown}')

In [ ]:
# 7. Risk helpers

def risk_level(probability: float, risk_bands: dict) -> str:
    if probability >= float(risk_bands['medium']):
        return 'High'
    if probability >= float(risk_bands['low']):
        return 'Medium'
    return 'Low'

def risk_score(probability: float) -> float:
    probability = min(max(float(probability), 0.0), 1.0)
    if probability < 0.10:
        score = (probability / 0.10) * 30
    elif probability < 0.20:
        score = 30 + ((probability - 0.10) / 0.10) * 40
    else:
        score = 70 + ((probability - 0.20) / 0.80) * 30
    return round(min(max(score, 0.0), 100.0), 2)

def top_risk_factors(model, row, customer_data, top_n=5):
    dmatrix = xgb.DMatrix(row, enable_categorical=True)
    contributions = model.get_booster().predict(dmatrix, pred_contribs=True)
    shap_values = np.asarray(contributions)[0, :-1]
    positive = [(f, float(v)) for f, v in zip(row.columns.tolist(), shap_values) if v > 0]
    positive.sort(key=lambda item: item[1], reverse=True)
    return [{'factor': f, 'value': json_safe(customer_data[f])} for f, _ in positive[:top_n]]

In [ ]:
# 8. Load models

def load_model1():
    global MODEL1, CALIBRATOR, METADATA
    print('Loading Model 1...')
    upload_file_if_missing(MODEL1_PATH)
    upload_file_if_missing(CALIBRATOR_PATH)
    upload_file_if_missing(METADATA_PATH)

    print('Loading Model 1 metadata...')
    with open(METADATA_PATH, 'r', encoding='utf-8') as f:
        METADATA = json.load(f)
    print('Loading Model 1 calibrator...')
    CALIBRATOR = joblib.load(CALIBRATOR_PATH)
    print('Loading XGBoost model...')
    MODEL1 = XGBClassifier()
    MODEL1.load_model(MODEL1_PATH)
    print('Model 1 loaded successfully')

def load_model2(model_path=None):
    global MODEL2, MODEL2_PATH
    if model_path:
        MODEL2_PATH = model_path
    print('Loading Model 2...')
    upload_file_if_missing(MODEL2_PATH)
    print('Loading GGUF model. This can take a few minutes...')
    MODEL2 = Llama(
        model_path=MODEL2_PATH,
        n_ctx=MODEL2_N_CTX,
        n_threads=MODEL2_N_THREADS,
        n_batch=MODEL2_N_BATCH,
        n_gpu_layers=0,
        verbose=False,
    )
    print('Model 2 loaded successfully')

print('Starting model loading step...')
if USE_MODEL1:
    load_model1()
else:
    print('Model 1 disabled')
if USE_MODEL2:
    load_model2()
else:
    print('Model 2 disabled')
print('All enabled models are ready')

In [ ]:
# 9. Predict functions

def predict_model1(customer_data: dict, threshold: Optional[float] = None) -> dict:
    if MODEL1 is None:
        raise RuntimeError('Model 1 is not loaded')
    features = METADATA['features']
    threshold = float(METADATA['threshold'] if threshold is None else threshold)
    validate_model1_input(customer_data, features)
    row = prepare_feature_row(customer_data, METADATA)
    raw_probability = float(MODEL1.predict_proba(row)[0, 1])
    probability = float(CALIBRATOR.predict([raw_probability])[0])
    return {
        'churn_probability': round(probability * 100, 2),
        'risk_score': risk_score(probability),
        'churn_prediction': 'Yes' if probability >= threshold else 'No',
        'risk_level': risk_level(probability, METADATA['risk_bands']),
        'top_risk_factors': top_risk_factors(MODEL1, row, customer_data),
    }

def predict_model2(payload: dict) -> dict:
    if MODEL2 is None:
        raise RuntimeError('Model 2 is not loaded')
    system_prompt = (
        'You are a retention intelligence assistant for a retail bank. '
        'Return ONLY a JSON object with exactly these two keys: why and next_actions. '
        'Both values must be arrays of short strings. '
        'Put the explanation only in why. '
        'Put every recommendation only in next_actions. '
        'Never combine the keys or put recommendations in why. '
        'Example: {"why": ["Risk increased after repeated transaction declines."], '
        '"next_actions": ["Contact the customer within 24 hours.", "Offer transaction support."]}'
    )
    with MODEL2_LOCK:
        output = MODEL2.create_chat_completion(
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': json.dumps(payload)},
            ],
            response_format={'type': 'json_object'},
            temperature=MODEL2_TEMPERATURE,
            max_tokens=MODEL2_MAX_TOKENS,
        )
    text = output['choices'][0]['message']['content']
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {'why': [text], 'next_actions': []}

def build_model2_payload(customer, model1_output, extra_context=None):
    extra_context = extra_context or {}
    return {
        'type': 'individual',
        'model1_output': model1_output,
        'customer_profile': extra_context.get('customer_profile', {}),
        'current_snapshot': customer,
        'trend_last_3_months': extra_context.get('trend_last_3_months', {}),
        'recent_complaint_text': extra_context.get('recent_complaint_text'),
        'risk_group': extra_context.get('risk_group', 'unknown'),
    }

In [ ]:
# 10. API

class Model1Request(BaseModel):
    customer: dict
    threshold: Optional[float] = None

class Model2Request(BaseModel):
    payload: dict

class BothRequest(BaseModel):
    customer: dict
    extra_context: Optional[dict] = None
    threshold: Optional[float] = None

class LoadModel2Request(BaseModel):
    model_path: str

app = FastAPI(title='Customer Retention Colab API')

@app.get('/health')
def health():
    return {'ok': True, 'model1_loaded': MODEL1 is not None, 'model2_loaded': MODEL2 is not None}

@app.post('/predict/model1')
def api_model1(request: Model1Request):
    try:
        return {'model1': predict_model1(request.customer, request.threshold)}
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))

@app.post('/predict/model2')
def api_model2(request: Model2Request):
    try:
        return {'model2': predict_model2(request.payload)}
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))

@app.post('/predict/both')
def api_both(request: BothRequest):
    try:
        model1 = predict_model1(request.customer, request.threshold)
        payload = build_model2_payload(request.customer, model1, request.extra_context)
        model2 = predict_model2(payload)
        return {'model1': model1, 'model2_input': payload, 'model2': model2}
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))

@app.post('/admin/load-model2')
def api_load_model2(request: LoadModel2Request):
    try:
        load_model2(request.model_path)
        return {'ok': True, 'model2_path': MODEL2_PATH}
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))

In [ ]:
# 11. Start API with Cloudflare Tunnel

TUNNEL_PROCESS = None

def ensure_cloudflared():
    path = Path('/content/cloudflared')
    if path.exists():
        return str(path)
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    print('Downloading cloudflared...')
    urllib.request.urlretrieve(url, path)
    path.chmod(path.stat().st_mode | stat.S_IEXEC)
    return str(path)

def start_cloudflare_tunnel(local_url):
    global TUNNEL_PROCESS
    print('Starting Cloudflare Tunnel...')
    TUNNEL_PROCESS = subprocess.Popen(
        [ensure_cloudflared(), 'tunnel', '--url', local_url, '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    pattern = re.compile(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com')
    deadline = time.time() + 60
    while time.time() < deadline:
        line = TUNNEL_PROCESS.stdout.readline()
        if line:
            print(line.strip())
            match = pattern.search(line)
            if match:
                return match.group(0)
    raise RuntimeError('Cloudflare tunnel did not return a public URL')

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=API_PORT, log_level='info')

print('Starting FastAPI server...')
threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print('FastAPI server is running')

local_url = f'http://127.0.0.1:{API_PORT}'
api_url = start_cloudflare_tunnel(local_url)

print('API URL:', api_url)
print('Docs:', api_url + '/docs')